In [9]:
import re
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# -----------
# Helpers: Parsin e Limpeza
# -----------

PT_MONTHS = {
    # pt -> en (abreviações comuns)
    "jan": "Jan", "fev": "Feb", "mar": "Mar", "abr": "Apr", "mai": "May", "jun": "Jun",
    "jul": "Jul", "ago": "Aug", "set": "Sep", "out": "Oct", "nov": "Nov", "dez": "Dec",
}

def _normalize_months_pt_en(s: pd.Series) -> pd.Series:
    """Converte meses pt-br (com/sem ponto) para meses EN, mantendo formato tipo 'jun. 12, 2024 16:05'."""
    if s.dtype != "object":
        s = s.astype("object")
    out = s.fillna("").astype(str)

    # normaliza espacos e remove duplicidades
    out = out.str.strip()

    # substitui "jun." -> "jun" (tirar ponto)
    out = out.str.replace(r"\b(jan|fev|mar|abr|mai|jun|jul|ago|set|out|nov|dez)\.\b", r"\1", regex=True, flags=re.IGNORECASE)

    # troca meses pt pa en (case-insensitive)
    for pt, en in PT_MONTHS.items():
        out = out.str.replace(rf"\b{pt}\b", en, regex=True, flags=re.IGNORECASE)

    # devolve vazios como NaN
    out = out.replace({"": np.nan})
    return out

def parse_datetime_mixed(series: pd.Series) -> pd.Series:
    """Parse robusto para datas tipo 'May 5, 2024 7:59 am' e 'jun. 12, 2024 16:05'."""
    s = _normalize_months_pt_en(series)
    # pandas lida bem com AM/PM e 24h após normalizar meses
    dt = pd.to_datetime(s, errors="coerce", infer_datetime_format=True)
    return dt

def to_number_pt(series: pd.Series) -> pd.Series:
    """Converte números com vírgula decimal e possíveis separadores de milhar."""
    if series.dtype == "number":
        return series
    s = series.astype("object").where(series.notna(), None)
    s = pd.Series(s)

    def _conv(x):
        if x is None:
            return np.nan
        if isinstance(x, (int, float, np.number)):
            return float(x)
        txt = str(x).strip()
        if txt == "" or txt.lower() in {"nan", "none"}:
            return np.nan

        # remove espaços
        txt = txt.replace(" ", "")

        # se tem '.' e ',' assume padrão BR: '.' milhar, ',' decimal
        if "," in txt and "." in txt:
            txt = txt.replace(".", "").replace(",", ".")
        # se só tem ',' assume decimal
        elif "," in txt:
            txt = txt.replace(",", ".")
        # caso contrário mantém como está

        try:
            return float(txt)
        except:
            return np.nan

    return s.map(_conv)

def to_bool_sim_nao(series: pd.Series) -> pd.Series:
    """Mapeia sim/não/nao/yes/no para booleano; mantém NaN se vazio."""
    mapping = {
        "sim": True, "s": True, "yes": True, "y": True, "true": True, "1": True,
        "não": False, "nao": False, "n": False, "no": False, "false": False, "0": False
    }
    s = series.astype("object")
    s = s.where(series.notna(), None)

    def _map(x):
        if x is None:
            return np.nan
        txt = str(x).strip().lower()
        if txt == "":
            return np.nan
        return mapping.get(txt, np.nan)

    return s.map(_map)

# -----------
# Auditoria 
# -----------

def basic_profile(df: pd.DataFrame, name: str, key_cols=None, date_cols=None, num_cols=None) -> dict:
    key_cols = key_cols or []
    date_cols = date_cols or []
    num_cols = num_cols or []

    profile = {}
    profile["table"] = name
    profile["rows"] = len(df)
    profile["cols"] = df.shape[1]

    # Null rate geral por coluna
    null_rate = (df.isna().mean().sort_values(ascending=False) * 100).round(2)
    profile["null_rate"] = null_rate

    #duplicidade em chaves
    dup_info = {}
    for k in key_cols:
        if k in df.columns:
            dup_info[k] = {
                "n_unique": int(df[k].nunique(dropna=True)),
                "n_null": int(df[k].isna().sum()),
                "n_duplicates": int(df.duplicated(subset=[k]).sum()),
            }
    profile["keys"] = dup_info

    # Datas: parse + falhas
    date_info = {}
    for c in date_cols:
        if c in df.columns:
            parsed = parse_datetime_mixed(df[c])
            n_total = df[c].notna().sum()
            n_failed = parsed.isna().sum() - df[c].isna().sum()
            date_info[c] = {
                "non_null": int(n_total),
                "failed_parse": int(max(n_failed, 0)),
                "min": str(parsed.min()) if parsed.notna().any() else None,
                "max": str(parsed.max()) if parsed.notna().any() else None,
            }
    profile["dates"] = date_info

    # Números: conversão + falhas
    num_info = {}
    for c in num_cols:
        if c in df.columns:
            conv = to_number_pt(df[c])
            n_non_null = int(df[c].notna().sum())
            n_failed = int(((df[c].notna()) & (conv.isna())).sum())
            num_info[c] = {
                "non_null": n_non_null,
                "failed_numeric_parse": n_failed,
                "min": float(np.nanmin(conv.values)) if np.isfinite(conv.values).any() else None,
                "max": float(np.nanmax(conv.values)) if np.isfinite(conv.values).any() else None,
            }
    profile["numbers"] = num_info

    return profile

def save_audit(profile: dict, out_prefix: str):
    # 1) resumo
    summary = {
        "table": profile["table"],
        "rows": profile["rows"],
        "cols": profile["cols"],
    }
    pd.DataFrame([summary]).to_csv(f"{out_prefix}_summary.csv", index=False)

    # 2) null rate
    profile["null_rate"].reset_index().rename(columns={"index": "column", 0: "null_rate_pct"}) \
        .to_csv(f"{out_prefix}_null_rate.csv", index=False)

    # 3) keys
    if profile["keys"]:
        rows = []
        for k, v in profile["keys"].items():
            rows.append({"key": k, **v})
        pd.DataFrame(rows).to_csv(f"{out_prefix}_keys.csv", index=False)

    # 4) dates
    if profile["dates"]:
        rows = []
        for c, v in profile["dates"].items():
            rows.append({"date_col": c, **v})
        pd.DataFrame(rows).to_csv(f"{out_prefix}_dates.csv", index=False)

    # 5) numbers
    if profile["numbers"]:
        rows = []
        for c, v in profile["numbers"].items():
            rows.append({"num_col": c, **v})
        pd.DataFrame(rows).to_csv(f"{out_prefix}_numbers.csv", index=False)

# -------------
# EXECUCAO
# -------------

t1 = pd.read_csv("../data/raw/export_-JO-O--An-lise-Full_2025-12-16_16-57-32.csv", dtype=str)
t2 = pd.read_csv("../data/raw/export_View---Vencimentos-CESKGPF_2025-12-16_11-36-56 - export_View---Vencimentos-CESKGPF_2025-12-16_11-36-56.csv", dtype=str)

# Colunas de interesse (com base no que você descreveu)
T1_DATE_COLS = ["[PAGO] Data do Pagamento", "Data Analise/Proposta", "Creation Date", "Modified Date"]
T2_DATE_COLS = ["[PAGO] Data Pagamento", "Creation Date", "Data de vencimento"]

T1_NUM_COLS = ["[PAGO] Valor Emprestado", "Taxa de Juros", "Taxa de Juros Rebate", "Score Serasa"]
T2_NUM_COLS = ["[PAGO] Valor Pago", "Valor Parcela", "Valor total"]

T1_KEYS = ["proposal", "unique id", "CPF Proposta"]
T2_KEYS = ["unique id", "ID", "Slug"]

DT_PAYMENT_COLS = [f"DT|Payment {i}" for i in range(1, 7)]

def quick_dt_payment_fill(df):
    cols = [c for c in DT_PAYMENT_COLS if c in df.columns]
    out = []
    for c in cols:
        out.append({
            "col": c,
            "non_null": int(df[c].notna().sum()),
            "null": int(df[c].isna().sum()),
        })
    return pd.DataFrame(out)

def quick_match_rate_dtpayments(t1, t2):
    """Pré-check: quantos IDs em DT|Payment aparecem em unique id da tabela2 (sem explodir)."""
    if "unique id" not in t2.columns:
        return None
    t2_ids = set(t2["unique id"].dropna().astype(str).unique())

    rows = []
    for c in DT_PAYMENT_COLS:
        if c not in t1.columns:
            continue
        ids = t1[c].dropna().astype(str)
        n = len(ids)
        m = sum(i in t2_ids for i in ids)
        rows.append({"dt_col": c, "n_ids": n, "n_matched": m, "match_rate_pct": round((m / n) * 100, 2) if n else None})
    return pd.DataFrame(rows)

prof1 = basic_profile(t1, "tabela1", key_cols=T1_KEYS, date_cols=T1_DATE_COLS, num_cols=T1_NUM_COLS)
prof2 = basic_profile(t2, "tabela2", key_cols=T2_KEYS, date_cols=T2_DATE_COLS, num_cols=T2_NUM_COLS)
save_audit(prof1, "../reports/audit/tabela1")
save_audit(prof2, "../reports/audit/tabela2")

quick_dt_payment_fill(t1).to_csv("../reports/audit/tabela1_dt_payment_fill.csv", index=False)
mr = quick_match_rate_dtpayments(t1, t2)
if mr is not None:
    mr.to_csv("../reports/audit/dt_payment_match_rate.csv", index=False)